# Big Data Log Analysis
**PySpark 3.5 on Databricks** — processing 1M+ log rows with optimised distributed pipelines.

---
**Sections:**
1. Setup & Configuration
2. Data Ingestion
3. Distributed Processing & Aggregations
4. Anomaly Detection
5. Results & Visualisation

## 1. Setup & Configuration

In [ ]:
# Install dependencies (Databricks cluster already has PySpark)
%pip install -q pytest

import sys, os
sys.path.insert(0, '/Workspace/Repos/<your-repo>/big-data-log-analysis/src')

# Configuration
LOG_INPUT_PATH  = 'dbfs:/logs/prod/logs.csv'   # update to your DBFS path
LOG_OUTPUT_PATH = 'dbfs:/results/log-analysis'

print('Setup complete')

In [ ]:
from pyspark.sql import SparkSession

# On Databricks the session already exists — we just configure it
spark = SparkSession.builder.getOrCreate()

# Key performance configurations
spark.conf.set('spark.sql.adaptive.enabled',                        'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled',     'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled',               'true')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold',              '50MB')
spark.conf.set('spark.sql.shuffle.partitions',                      '200')

print(f'Spark version: {spark.version}')
print(f'Executors: {spark.sparkContext.defaultParallelism}')

## 2. Data Ingestion

In [ ]:
import time
from ingestion import load_logs

t0 = time.time()
df = load_logs(spark, LOG_INPUT_PATH)

print(f'\nIngestion time: {time.time()-t0:.1f}s')
print(f'Partitions    : {df.rdd.getNumPartitions()}')
df.printSchema()

In [ ]:
# Preview — first 5 rows
display(df.limit(5))

## 3. Distributed Processing & Aggregations

In [ ]:
from processing import run_processing

t1 = time.time()
results = run_processing(df)
print(f'Processing time: {time.time()-t1:.1f}s')

In [ ]:
print('=== Per-service KPIs ===')
display(results['by_service'])

In [ ]:
print('=== Top 10 Slowest Endpoints ===')
display(results['slow_endpoints'])

In [ ]:
print('=== Status Code Distribution ===')
display(results['status_dist'])

## 4. Anomaly Detection

In [ ]:
from processing import enrich, repartition_by_service
from anomaly_detection import run_anomaly_detection

df_enriched = repartition_by_service(enrich(df))

t2 = time.time()
anomalies = run_anomaly_detection(df_enriched)
print(f'Detection time: {time.time()-t2:.1f}s')

In [ ]:
print('=== Consolidated Anomaly Report ===')
display(anomalies['report'])

In [ ]:
print('=== Error Spikes (5-min windows) ===')
display(anomalies['error_spikes'])

In [ ]:
print('=== Brute-force / DDoS Suspects ===')
display(anomalies['brute_force'])

## 5. Save Results to Delta Lake

In [ ]:
all_results = {**results, **anomalies}

for name, result_df in all_results.items():
    path = f'{LOG_OUTPUT_PATH}/{name}.parquet'
    result_df.write.mode('overwrite').parquet(path)
    print(f'Saved {name} → {path}')

print('\nAll results saved.')